### Imports

In [1]:
import math
import hashlib
import random
from pathlib import Path

import torch
import pandas as pd
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    LogitsProcessor,
    LogitsProcessorList,
)

### Part a)

 a) Implement the green/red-list watermark. I suggest doing so by making it a
HuggingFace LogitsProcessor (see the stub below) . Suggested defaults: γ = 0.5
(green-list fraction), δ = 2.0 (logit bias), k = 1 (hash context length). Generate
watermarked responses for the Dolly prompts and implement the detector that returns a z-score and p-value.

def GreenListWatermarkProcessor

In [2]:
# @title
class GreenListWatermarkProcessor(LogitsProcessor):
    def __init__(self, vocab_size, gamma=0.5, delta=2.0, k=1):
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.delta = delta
        self.k = k
        self.green_size = int(gamma * vocab_size)

    def _seed_from_context(self, context_tokens):
        context_bytes = ",".join(map(str, context_tokens)).encode("utf-8")
        digest = hashlib.sha256(context_bytes).digest()
        return int.from_bytes(digest[:8], byteorder="big", signed=False)

    def _green_mask(self, context_tokens, device):
        seed = self._seed_from_context(context_tokens)

        generator = torch.Generator(device="cpu")
        generator.manual_seed(seed)

        perm = torch.randperm(self.vocab_size, generator=generator)
        green_tokens = perm[: self.green_size].to(device)

        mask = torch.zeros(self.vocab_size, dtype=torch.bool, device=device)
        mask[green_tokens] = True

        return mask

    def __call__(self, input_ids, logits):
        # input_ids: (batch, seq_len)
        # logits: (batch, vocab_size)

        for batch_idx in range(input_ids.shape[0]):
            context = input_ids[batch_idx, -self.k :].tolist()
            green_mask = self._green_mask(context, logits.device)

            logits[batch_idx, green_mask] += self.delta

        return logits

def detect_greenlist_watermark

In [3]:
# @title
def detect_greenlist_watermark(
    text,
    tokenizer,
    vocab_size,
    gamma=0.5,
    k=1,
):
    token_ids = tokenizer(text, return_tensors="pt")["input_ids"][0].tolist()

    processor = GreenListWatermarkProcessor(
        vocab_size=vocab_size,
        gamma=gamma,
        delta=0.0,
        k=k,
    )

    green_count = 0
    total_count = 0

    for i in range(k, len(token_ids)):
        context = token_ids[i - k : i]
        token = token_ids[i]

        green_mask = processor._green_mask(context, device="cpu")

        if green_mask[token].item():
            green_count += 1

        total_count += 1

    expected = gamma * total_count
    variance = total_count * gamma * (1 - gamma)

    z_score = (green_count - expected) / math.sqrt(variance)

    # one-sided p-value: probability of seeing this many or more green tokens
    p_value = 0.5 * math.erfc(z_score / math.sqrt(2))

    return {
        "green_count": green_count,
        "total_count": total_count,
        "green_fraction": green_count / total_count if total_count > 0 else 0,
        "z_score": z_score,
        "p_value": p_value,
    }

def download_model_weights

In [4]:
# @title
def download_model_weights(
    model_id: str = "Qwen/Qwen3-4B",
    model_path: str = "./Qwen3-4B",
    required_files: list[str] | None = None,
) -> Path:
    """
    Download model weights from Hugging Face if they are not already present locally.

    Args:
        model_id: Hugging Face model repository ID.
        model_path: Local directory where the model should be stored.
        required_files: Files used to check whether the model already exists.

    Returns:
        Path to the local model directory.
    """
    if required_files is None:
        required_files = [
            "config.json",
            "tokenizer.json",
        ]

    model_path = Path(model_path)

    model_exists = all(
        (model_path / file).exists()
        for file in required_files
    )

    if not model_exists:
        print(f"Model not found locally at {model_path}")
        print(f"Downloading {model_id} ...")

        snapshot_download(
            repo_id=model_id,
            local_dir=str(model_path),
            local_dir_use_symlinks=False,
        )

        print("Download complete.")
    else:
        print(f"Using existing local model at {model_path}")

    return model_path

def load_model_weights

In [5]:
# @title
def load_model_weights(model_path: str = "./Qwen3-4B"):
    """
    Load tokenizer and model weights from a local model directory.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
        trust_remote_code=True,
    )

    if device == "cpu":
        model = model.to(device)

    model.eval()

    return model, tokenizer, device


def generate_response(
    prompt: str,
    model,
    tokenizer,
    watermarked: bool = True,
    gamma: float = 0.5,
    delta: float = 2.0,
    k: int = 1,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.95,
):
    """
    Generate a response from the model.

    If watermarked=True, applies the GreenListWatermarkProcessor.
    If watermarked=False, generates normally.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    generate_kwargs = {
        **inputs,
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": temperature,
        "top_p": top_p,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if watermarked:
        watermark_processor = GreenListWatermarkProcessor(
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            delta=delta,
            k=k,
        )

        generate_kwargs["logits_processor"] = LogitsProcessorList(
            [watermark_processor]
        )

    with torch.inference_mode():
        output_ids = model.generate(**generate_kwargs)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][prompt_len:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return response


def evaluate_watermark(
    text: str,
    tokenizer,
    vocab_size: int,
    gamma: float = 0.5,
    k: int = 1,
):
    """
    Evaluate whether a text contains the green-list watermark.
    """
    detection_result = detect_greenlist_watermark(
        text=text,
        tokenizer=tokenizer,
        vocab_size=vocab_size,
        gamma=gamma,
        k=k,
    )

    return detection_result

In [6]:
# Load Dolly dataset
dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train",
)

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [9]:
MODEL_PATH = "./Qwen3-4B"
download_model_weights(
    model_id="Qwen/Qwen3-4B",
    model_path=MODEL_PATH,
)

# Load model + tokenizer
model, tokenizer, device = load_model_weights(MODEL_PATH)

Model not found locally at Qwen3-4B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Download complete.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# Prepare one prompt
example = dataset[0]

instruction = example["instruction"]
context = example["context"]

if context and len(context.strip()) > 0:
    prompt = f"""Instruction:
{instruction}

Context:
{context}

Response:"""
else:
    prompt = f"""Instruction:
{instruction}

Response:"""

print("PROMPT:")
print(prompt)

# Generate response
watermarked_response = generate_response(
    prompt=prompt,
    model=model,
    tokenizer=tokenizer,
    watermarked=True,
    gamma=0.5,
    delta=2.0,
    k=1,
)

print("\nWATERMARKED RESPONSE:")
print(watermarked_response)

detection_result = evaluate_watermark(
    text=watermarked_response,
    tokenizer=tokenizer,
    vocab_size=model.config.vocab_size,
    gamma=0.5,
    k=1,
)

print("\nWATERMARK DETECTION:")
print(detection_result)

Using existing local model at Qwen3-4B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

PROMPT:
Instruction:
When did Virgin Australia start operating?

Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

Response:

WATERMARKED RESPONSE:
 The answer is 31 August 2000.

But the user might be confused because of the name. The original name was Virgin Blue, but it was later renamed to Virgin Australia. The correct answer is 31 August 2000.

I need to make sure the user understands that even though the name changed, the start date remains the same.
The answer is 31 August 2000. Virgin Australia originally began oper

### Part b)

Evaluate robustness via a sanity check and paraphrase attack:

○ Sanity check. Run your detector on human responses and unwatermarked
Qwen3-4B responses; both should yield z-scores ≈ 0. Confirm watermarked
generations have meaningfully positive z-scores.

○ Paraphrase attack. Use an LLM to rewrite each watermarked response. Report
z-scores/p-values before and after.

#### b) - sanity check

def build_dolly_prompt

In [11]:
# @title
def build_dolly_prompt(example):
    """
    Build the same prompt format you already use for Dolly examples.
    """
    instruction = example["instruction"]
    context = example["context"]

    if context and len(context.strip()) > 0:
        prompt = f"""Instruction:
{instruction}

Context:
{context}

Response:"""
    else:
        prompt = f"""Instruction:
{instruction}

Response:"""

    return prompt

def evaluate_human_and_nonwatermarked_responses

In [20]:
# @title
def evaluate_human_and_nonwatermarked_responses(
    model,
    tokenizer,
    dataset_name: str = "databricks/databricks-dolly-15k",
    split: str = "train",
    n_samples: int = 10,
    seed: int = 42,
    gamma: float = 0.5,
    k: int = 1,
    max_new_tokens: int = 256,
    output_path: str = "dolly_human_and_nonwatermarked_responses.jsonl",
):
    """
    Samples Dolly examples, evaluates:
    1. Human-written Dolly responses
    2. Non-watermarked model-generated responses

    using the green-list watermark detector.

    Saves questions, answers, and detector results to a JSONL file so they can
    be loaded later in another script.
    """

    dataset = load_dataset(dataset_name, split=split)

    random.seed(seed)
    sample_indices = random.sample(range(len(dataset)), n_samples)

    results = []

    iter_num = 1
    for idx in sample_indices:
        print("Iter num ", iter_num)
        example = dataset[idx]

        prompt = build_dolly_prompt(example)
        human_response = example["response"]

        # Generate non-watermarked Qwen response
        nonwatermarked_response = generate_response(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            watermarked=False,
            gamma=gamma,
            k=k,
            max_new_tokens=max_new_tokens,
        )

        # Evaluate human response
        human_detection = evaluate_watermark(
            text=human_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        # Evaluate non-watermarked model response
        nonwatermarked_detection = evaluate_watermark(
            text=nonwatermarked_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        results.append({
            "dataset_index": idx,

            # Question / prompt data
            "instruction": example["instruction"],
            "context": example["context"],
            "prompt": prompt,

            # Answers
            "human_response": human_response,
            "nonwatermarked_response": nonwatermarked_response,

            # Human response watermark detection
            "human_green_count": human_detection["green_count"],
            "human_total_count": human_detection["total_count"],
            "human_green_fraction": human_detection["green_fraction"],
            "human_z_score": human_detection["z_score"],
            "human_p_value": human_detection["p_value"],

            # Non-watermarked response watermark detection
            "nonwatermarked_green_count": nonwatermarked_detection["green_count"],
            "nonwatermarked_total_count": nonwatermarked_detection["total_count"],
            "nonwatermarked_green_fraction": nonwatermarked_detection["green_fraction"],
            "nonwatermarked_z_score": nonwatermarked_detection["z_score"],
            "nonwatermarked_p_value": nonwatermarked_detection["p_value"],
        })

    df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(f"Saved {len(df)} examples to: {output_path}")

    return df

In [17]:
sanity_df = evaluate_human_and_nonwatermarked_responses(
    model=model,
    tokenizer=tokenizer,
    n_samples=10,
    seed=42,
    gamma=0.5,
    k=1,
    max_new_tokens=256,
    output_path="outputs/dolly_sanity_check.jsonl",
)

sanity_df[
    [
        "dataset_index",
        "human_z_score",
        "human_p_value",
        "nonwatermarked_z_score",
        "nonwatermarked_p_value",
    ]
]

Saved 10 examples to: outputs/dolly_sanity_check.jsonl


,dataset_index,human_z_score,human_p_value,nonwatermarked_z_score,nonwatermarked_p_value
0,10476,0.000000,0.500000,2.818009,0.002416
1,1824,-1.897367,0.971110,-1.189826,0.882943
2,409,-0.577350,0.718149,-0.313112,0.622902
3,12149,0.000000,0.500000,-0.438357,0.669436
4,4506,0.000000,0.500000,-0.187867,0.574510
5,4012,0.306987,0.379427,-0.062622,0.524966
6,3657,2.121320,0.016947,-0.688847,0.754540
7,2286,-0.277350,0.609244,-0.563602,0.713487
8,12066,2.921187,0.001744,-0.688847,0.754540
9,1679,1.047645,0.147401,-1.565561,0.941274


In [18]:
sanity_df = pd.read_json(
    "outputs/dolly_sanity_check.jsonl",
    lines=True,
)

sanity_df.head()

,dataset_index,instruction,context,prompt,human_response,nonwatermarked_response,human_green_count,human_total_count,human_green_fraction,human_z_score,human_p_value,nonwatermarked_green_count,nonwatermarked_total_count,nonwatermarked_green_fraction,nonwatermarked_z_score,nonwatermarked_p_value
0,10476,What athlete created the 'beast quake' for the...,,Instruction:\nWhat athlete created the 'beast ...,Marshan Lynch,The 'beast quake' is a term used to describe ...,1,2,0.500000,0.000000,0.500000,150,255,0.588235,2.818009,0.002416
1,1824,Who wrote Democracy in America?,,Instruction:\nWho wrote Democracy in America?\...,Alexis de Tocqueville wrote Democracy in America,\n\nThe question is asking about the author o...,2,10,0.200000,-1.897367,0.971110,118,255,0.462745,-1.189826,0.882943
2,409,"What links Brazil, Uruguay, Mozambique and Angola",,"Instruction:\nWhat links Brazil, Uruguay, Moza...",Colonies of Portugal,"\nOkay, so I need to figure out what links Br...",1,3,0.333333,-0.577350,0.718149,125,255,0.490196,-0.313112,0.622902
3,12149,Extract the causes of income inequality in the...,"According to CBO (and others), the precise rea...",Instruction:\nExtract the causes of income ine...,The text lists the following as causes of inco...,"decline of labor unions, globalization, skill...",36,72,0.500000,0.000000,0.500000,124,255,0.486275,-0.438357,0.669436
4,4506,Do you really need mobile phone?,,Instruction:\nDo you really need mobile phone?...,We do not really need a mobile phone to live.,\nI don't need a mobile phone. I can use my c...,5,10,0.500000,0.000000,0.500000,126,255,0.494118,-0.187867,0.574510


In [19]:
print("Human responses:")
print(sanity_df["human_z_score"].describe())

print("\nNon-watermarked model responses:")
print(sanity_df["nonwatermarked_z_score"].describe())

Human responses:
count    10.000000
mean      0.364507
std       1.370593
min      -1.897367
25%      -0.208013
50%       0.000000
75%       0.862481
max       2.921187
Name: human_z_score, dtype: float64

Non-watermarked model responses:
count    10.000000
mean     -0.288063
std       1.182000
min      -1.565561
25%      -0.688847
50%      -0.500979
75%      -0.219179
max       2.818009
Name: nonwatermarked_z_score, dtype: float64


#### b) - Paraphrase attack

def run_part_b_sanity_check

In [27]:
# @title
def run_part_b_sanity_check(
    model,
    tokenizer,
    dataset_name: str = "databricks/databricks-dolly-15k",
    split: str = "train",
    n_samples: int = 10,
    seed: int = 42,
    gamma: float = 0.5,
    delta: float = 2.0,
    k: int = 1,
    max_new_tokens: int = 256,
    output_path: str = "outputs/part_b_sanity_check.jsonl",
):
    """
    Part b sanity check.

    Evaluates watermark detector on:
    1. Human Dolly responses
    2. Non-watermarked Qwen responses
    3. Watermarked Qwen responses

    Saves all prompts, answers, z-scores, and p-values to a JSONL file.
    """

    dataset = load_dataset(dataset_name, split=split)

    random.seed(seed)
    sample_indices = random.sample(range(len(dataset)), n_samples)

    results = []

    for i, idx in enumerate(sample_indices):
        print(f"Processing sample {i + 1}/{n_samples}")

        example = dataset[idx]
        prompt = build_dolly_prompt(example)

        human_response = example["response"]

        nonwatermarked_response = generate_response(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            watermarked=False,
            gamma=gamma,
            delta=delta,
            k=k,
            max_new_tokens=max_new_tokens,
        )

        watermarked_response = generate_response(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            watermarked=True,
            gamma=gamma,
            delta=delta,
            k=k,
            max_new_tokens=max_new_tokens,
        )

        human_detection = evaluate_watermark(
            text=human_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        nonwatermarked_detection = evaluate_watermark(
            text=nonwatermarked_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        watermarked_detection = evaluate_watermark(
            text=watermarked_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        results.append({
            "dataset_index": idx,
            "instruction": example["instruction"],
            "context": example["context"],
            "prompt": prompt,

            "human_response": human_response,
            "nonwatermarked_response": nonwatermarked_response,
            "watermarked_response": watermarked_response,

            "human_z_score": human_detection["z_score"],
            "human_p_value": human_detection["p_value"],
            "human_green_fraction": human_detection["green_fraction"],

            "nonwatermarked_z_score": nonwatermarked_detection["z_score"],
            "nonwatermarked_p_value": nonwatermarked_detection["p_value"],
            "nonwatermarked_green_fraction": nonwatermarked_detection["green_fraction"],

            "watermarked_z_score": watermarked_detection["z_score"],
            "watermarked_p_value": watermarked_detection["p_value"],
            "watermarked_green_fraction": watermarked_detection["green_fraction"],
        })

    sanity_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    sanity_df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(f"Saved sanity check results to: {output_path}")

    return sanity_df

def run_part_b_paraphrase_attack

In [28]:
# @title
def run_part_b_paraphrase_attack(
    sanity_df,
    model,
    tokenizer,
    gamma: float = 0.5,
    k: int = 1,
    paraphrase_max_new_tokens: int = 256,
    output_path: str = "outputs/part_b_paraphrase_attack.jsonl",
):
    """
    Part b paraphrase attack.

    Uses Qwen to rewrite each watermarked response.
    Reports detector z-scores and p-values before and after paraphrasing.
    """

    results = []

    for i, row in sanity_df.iterrows():
        print(f"Paraphrasing sample {i + 1}/{len(sanity_df)}")

        watermarked_response = row["watermarked_response"]

        before_detection = evaluate_watermark(
            text=watermarked_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        paraphrase_prompt = f"""Instruction:
Rewrite the following text in different words while preserving the original meaning.
Do not add new information.
Do not explain what you are doing.
Only output the rewritten text.

Text:
{watermarked_response}

Paraphrased response:"""

        paraphrased_response = generate_response(
            prompt=paraphrase_prompt,
            model=model,
            tokenizer=tokenizer,
            watermarked=False,
            max_new_tokens=paraphrase_max_new_tokens,
            temperature=0.7,
            top_p=0.95,
        )

        after_detection = evaluate_watermark(
            text=paraphrased_response,
            tokenizer=tokenizer,
            vocab_size=model.config.vocab_size,
            gamma=gamma,
            k=k,
        )

        result = row.to_dict()

        result.update({
            "paraphrased_response": paraphrased_response,

            "before_paraphrase_z_score": before_detection["z_score"],
            "before_paraphrase_p_value": before_detection["p_value"],
            "before_paraphrase_green_fraction": before_detection["green_fraction"],

            "after_paraphrase_z_score": after_detection["z_score"],
            "after_paraphrase_p_value": after_detection["p_value"],
            "after_paraphrase_green_fraction": after_detection["green_fraction"],
        })

        results.append(result)

    attack_df = pd.DataFrame(results)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    attack_df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(f"Saved paraphrase attack results to: {output_path}")

    return attack_df

In [ ]:
sanity_df = run_part_b_sanity_check(
    model=model,
    tokenizer=tokenizer,
    n_samples=10,
    seed=42,
    gamma=0.5,
    delta=2.0,
    k=1,
    max_new_tokens=256,
)

In [29]:
attack_df = run_part_b_paraphrase_attack(
    sanity_df=sanity_df,
    model=model,
    tokenizer=tokenizer,
    gamma=0.5,
    k=1,
    paraphrase_max_new_tokens=256,
)

Paraphrasing sample 1/10
Paraphrasing sample 2/10
Paraphrasing sample 3/10
Paraphrasing sample 4/10
Paraphrasing sample 5/10
Paraphrasing sample 6/10
Paraphrasing sample 7/10
Paraphrasing sample 8/10
Paraphrasing sample 9/10
Paraphrasing sample 10/10
Saved paraphrase attack results to: outputs/part_b_paraphrase_attack.jsonl


In [30]:
print("Sanity check z-score means:")
print("Human:", sanity_df["human_z_score"].mean())
print("Non-watermarked:", sanity_df["nonwatermarked_z_score"].mean())
print("Watermarked:", sanity_df["watermarked_z_score"].mean())

print("\nParaphrase attack z-score means:")
print("Before paraphrase:", attack_df["before_paraphrase_z_score"].mean())
print("After paraphrase:", attack_df["after_paraphrase_z_score"].mean())

attack_df[
    [
        "dataset_index",
        "before_paraphrase_z_score",
        "before_paraphrase_p_value",
        "after_paraphrase_z_score",
        "after_paraphrase_p_value",
    ]
]

Sanity check z-score means:
Human: 0.36450725032296183
Non-watermarked: 0.13776934403873284
Watermarked: 4.77182909806884

Paraphrase attack z-score means:
Before paraphrase: 4.77182909806884
After paraphrase: 0.9518609224494273


,dataset_index,before_paraphrase_z_score,before_paraphrase_p_value,after_paraphrase_z_score,after_paraphrase_p_value
0,10476,4.821927,7.108898e-07,3.569478,0.000179
1,1824,5.322906,5.106113e-08,0.313112,0.377098
2,409,4.947172,3.764974e-07,2.692764,0.003543
3,12149,1.690806,4.543697e-02,-0.313112,0.622902
4,4506,6.074376,6.223553e-10,0.563602,0.286513
5,4012,4.070458,2.346041e-05,0.814092,0.207796
6,3657,6.825845,4.370470e-12,-0.313112,0.622902
7,2286,3.193744,7.022033e-04,-0.438357,0.669436
8,12066,6.324865,1.267268e-10,1.690806,0.045437
9,1679,4.446192,4.370282e-06,0.939336,0.173779


### Part c)

Please, refer to the write-up document.